# 03 Data Preparation

In CRISP-DM Phase 3 wird aus dem rohen Datensatz ein sauberes, reproduzierbares Feature-Set aufgebaut,
das in Phase 4 (Modellierung) direkt eingesetzt werden kann.
Ziel dieses Notebooks ist keine Modellauswahl, sondern die Herstellung
methodisch korrekter Traings- und Testmengen sowie die Vorbereitung aller Features.

**Struktur:**
1. Setup
2. Datensatz laden
3. Stratifizierter Train/Test-Split
4. Feature-Typen definieren
5. Preprocessing-Pipeline
6. Transformierten Datensatz speichern

## 1. Setup

Alle Bibliotheken werden hier zentral importiert. Der globale Seed wird einmalig
festgelegt und in allen randomisierten Schritten (Split, SMOTE, Modelle) konsistent
verwendet — so ist das gesamte Notebook mit einem einzigen Wert reproduzierbar.

In [1]:
import importlib, subprocess, sys
for pkg in ['ucimlrepo', 'imbalanced-learn']:
    if importlib.util.find_spec(pkg.replace('-', '_')) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

SEED = 42

## 2. Datensatz laden

Der Datensatz wird erneut direkt aus dem UCI ML Repository geladen (ID 891).
Das stellt sicher, dass dieses Notebook unabhängig von Zwischenspeicherständen
aus Notebook 02 lauffähig ist und stets auf der Originalversion operiert.

In [2]:
dataset = fetch_ucirepo(id=891)
X = dataset.data.features.copy()
y = dataset.data.targets.squeeze().copy()

print(f"Features: {X.shape}")
print(f"Target:   {y.shape}")
print(f"Klassen:  {y.value_counts().to_dict()}")

Features: (253680, 21)
Target:   (253680,)
Klassen:  {0: 218334, 1: 35346}


## 3. Stratifizierter Train/Test-Split

Der Split wird als allererster Schritt vor jeder weiteren Transformation durchgeführt.
Das ist zwingend erforderlich, um Data Leakage zu verhindern: Skalierung, Imputation
und Oversampling werden ausschließlich auf dem Trainingsfold angepasst und dürfen
keinerlei Information aus dem Testfold einbeziehen.

**Warum stratifiziert?**
Bei einer Klassenverteilung von ca. 86 / 14 kann ein zufälliger Split die Prävalenz
in Trainings- und Testmenge systematisch verschieben — insbesondere in kleineren
Splits. Stratifizierung garantiert, dass beide Mengen dieselbe Klassenverteilung wie
der Gesamtdatensatz aufweisen.

**Warum 80/20?**
Bei ~253 680 Samples liefert ein 20-%-Testset bereits ~50 000 Beobachtungen —
ausreichend für stabile Metriken auch in Subgruppenanalysen. 80 % Trainingsdaten
stellen sicher, dass seltene Muster (positive Klasse, Subgruppen) gut repräsentiert sind.
Der Testset wird nach diesem Split **eingefroren** und bis zur finalen Evaluation
in Notebook 05 nicht mehr berührt.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

print(f"Train: {X_train.shape[0]:>7} Samples")
print(f"Test:  {X_test.shape[0]:>7} Samples")
print()
print("Target-Mean (Prävalenz positive Klasse):")
print(f"  Gesamt: {y.mean():.4f}")
print(f"  Train:  {y_train.mean():.4f}")
print(f"  Test:   {y_test.mean():.4f}")

# Absolute Zahlen pro Klasse
print("\nKlassenverteilung absolut:")
print(f"  Train — 0: {(y_train==0).sum():>6,}  1: {(y_train==1).sum():>5,}")
print(f"  Test  — 0: {(y_test==0).sum():>6,}  1: {(y_test==1).sum():>5,}")

Train:  202944 Samples
Test:    50736 Samples

Target-Mean (Prävalenz positive Klasse):
  Gesamt: 0.1393
  Train:  0.1393
  Test:   0.1393

Klassenverteilung absolut:
  Train — 0: 174,667  1: 28,277
  Test  — 0: 43,667  1: 7,069


## 4. Feature-Typen definieren

Die Features werden entsprechend ihrer semantischen Bedeutung gruppiert. Obwohl alle
Variablen technisch als Integer gespeichert sind, unterscheiden sie sich inhaltlich deutlich:
Einige Variablen sind binäre Indikatoren, andere sind ordinal kodierte Kategorien,
zwei Variablen zählen die Anzahl gesundheitlich eingeschränkter Tage und BMI ist das
einzige kontinuierlich interpretierbare numerische Feature.

Diese Gruppierung ist wichtig, weil verschiedene Modelltypen unterschiedlich auf
Feature-Skalierung, Ausreißer und kodierte Kategorien reagieren. Für eine erste
Baseline werden die Features möglichst nah an ihrer Originalform belassen. Dadurch
entsteht ein unveränderter Referenzpunkt, gegen den spätere Feature-Engineering-
Varianten verglichen werden können.

Die in Notebook 02 identifizierten möglichen Transformationen — insbesondere
BMI-Capping, Hurdle-Encoding für MentHlth und PhysHlth sowie Composite Features —
werden daher nicht sofort als finale Entscheidungen übernommen, sondern in den
folgenden Experimenten systematisch getestet. Ob eine Transformation beibehalten wird,
entscheidet sich anhand der Cross-Validation-Performance und nicht allein anhand der
explorativen Analyse.

In [ ]:
BINARY_COLS = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "Smoker",
    "Stroke",
    "HeartDiseaseorAttack",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "DiffWalk",
    "Sex",
]

ORDINAL_COLS = [
    "GenHlth",
    "Age",
    "Education",
    "Income",
]

COUNT_COLS = [
    "MentHlth",
    "PhysHlth",
]

NUMERIC_COLS = [
    "BMI",
]

ALL_FEATURES = BINARY_COLS + ORDINAL_COLS + COUNT_COLS + NUMERIC_COLS

print(f"Anzahl definierter Features: {len(ALL_FEATURES)}")
print(f"Anzahl Features im Datensatz: {X_train.shape[1]}")

print("\nNicht definierte Spalten:")
print(set(X_train.columns) - set(ALL_FEATURES))

print("\nDefiniert, aber nicht im Datensatz:")
print(set(ALL_FEATURES) - set(X_train.columns))

## 5. Transformationsfunktionen definieren

Die folgenden Hilfsfunktionen kapseln die in der EDA (Notebook 02) identifizierten
Feature-Transformationen. Sie werden hier nur **definiert**, noch nicht auf die
Trainingsdaten angewendet — das geschieht erst, wenn sie in eine sklearn-Pipeline
eingebaut werden. Dieses Vorgehen stellt sicher, dass kein Leakage entsteht und
jede Transformation nachvollziehbar dokumentiert ist.

Alle Funktionen arbeiten auf einer Kopie des DataFrames (`X.copy()`) und geben
einen DataFrame mit denselben Spalten zurück — außer dort, wo explizit neue
Spalten ergänzt oder Spalten entfernt werden.

Für BMI stehen zwei konkurrierende Varianten zur Verfügung, die im Modellierungs-
Notebook gegeneinander abgetestet werden:

- `cap_bmi`: behält BMI als kontinuierliche Variable, clippt nur die Extremwerte
  auf ein medizinisch plausibles Intervall. Geeignet für Modelle, die ordinale
  Abstände in BMI ausnutzen können (z. B. lineare Modelle, Gradient Boosting).
- `categorize_bmi`: überführt BMI in eine klinisch interpretierbare Ordinalskala
  (7 Stufen nach WHO-Klassifikation). Reduziert den Einfluss von Ausreißern stärker
  als Capping und ist robust gegenüber dem Messrauschen von Selbstauskünften.

In [ ]:
def cap_bmi(X, lower=18, upper=50):
    X = X.copy()
    X["BMI"] = X["BMI"].clip(lower=lower, upper=upper)
    return X


def categorize_bmi(X):
    # Klinische WHO-Einteilung: 0=Untergewicht, 1=Normalgewicht, 2=Übergewicht,
    # 3=Adipositas I, 4=Adipositas II, 5=Adipositas III, 6=Super-Adipositas
    # Quelle: https://dbknb.de/zentren/adipositaszentrum/adipositas
    X = X.copy()
    bins   = [0, 18.5, 25, 30, 35, 40, 50, 999]
    labels = [0, 1, 2, 3, 4, 5, 6]
    X["BMI"] = pd.cut(X["BMI"], bins=bins, labels=labels).astype(int)
    return X

In [ ]:
def hurdle_encode(X, cols=["MentHlth", "PhysHlth"]):
    X = X.copy()
    for col in cols:
        X[f"{col}_any"] = (X[col] > 0).astype(int)
        X[f"{col}_days"] = X[col]
        X = X.drop(columns=[col])
    return X

In [ ]:
def drop_low_info(X, cols=["CholCheck", "AnyHealthcare"]):
    X = X.copy()
    X = X.drop(columns=cols)
    return X

## 6. Sanity-Check der Transformationsfunktionen

Bevor die Funktionen in eine Pipeline eingebaut werden, wird ihr Verhalten
an kleinen Demonstrationsbeispielen überprüft. Dabei werden die Originaldaten
**nicht verändert** — alle Transformationen werden nur zu Prüfzwecken auf
temporäre Kopien angewendet.

Drei Checks werden durchgeführt:
1. **BMI-Capping:** Histogramm vor und nach `cap_bmi` zeigt, wie Extremwerte
   gekappt werden, ohne den Kernbereich der Verteilung zu berühren.
2. **BMI-Kategorisierung:** Balkendiagramm der Kategorienhäufigkeiten nach
   `categorize_bmi` prüft, ob die klinischen Grenzen korrekt angewendet werden
   und alle sieben Klassen (0–6) vorhanden sind.
3. **Hurdle-Encoding:** Die ersten Zeilen der neuen Spalten bestätigen, dass
   `_any` und `_days` korrekt aus den Originalwerten abgeleitet werden.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: BMI original
X_train["BMI"].plot.hist(bins=60, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("BMI — Original")
axes[0].set_xlabel("BMI")
axes[0].set_ylabel("Anzahl")

# Plot 2: BMI nach cap_bmi
X_capped = cap_bmi(X_train)
X_capped["BMI"].plot.hist(bins=60, ax=axes[1], color="darkorange", edgecolor="white")
n_affected = ((X_train["BMI"] < 18) | (X_train["BMI"] > 50)).sum()
axes[1].set_title(
    f"BMI — nach cap_bmi (18–50)\n{n_affected} Samples betroffen ({n_affected/len(X_train):.2%})"
)
axes[1].set_xlabel("BMI")

# Plot 3: BMI nach categorize_bmi
X_cat = categorize_bmi(X_train)
cat_labels = {
    0: "Untergewicht\n(<18.5)",
    1: "Normal\n(18.5–25)",
    2: "Übergewicht\n(25–30)",
    3: "Adipositas I\n(30–35)",
    4: "Adipositas II\n(35–40)",
    5: "Adipositas III\n(40–50)",
    6: "Super\n(>50)",
}
counts = X_cat["BMI"].value_counts().sort_index()
axes[2].bar(
    [cat_labels[i] for i in counts.index],
    counts.values,
    color="mediumseagreen",
    edgecolor="white",
)
axes[2].set_title("BMI — nach categorize_bmi (Klassen 0–6)")
axes[2].set_xlabel("Kategorie")
axes[2].set_ylabel("Anzahl")
axes[2].tick_params(axis="x", labelsize=7.5)

fig.suptitle(
    "Sanity-Check BMI-Transformationen: cap_bmi vs. categorize_bmi",
    fontsize=12,
)
plt.tight_layout()
plt.show()

print(f"Min/Max vor Capping:  {X_train['BMI'].min():.1f} / {X_train['BMI'].max():.1f}")
print(f"Min/Max nach Capping: {X_capped['BMI'].min():.1f} / {X_capped['BMI'].max():.1f}")
print(f"\nKategorienhäufigkeiten nach categorize_bmi:")
print(counts.rename(index=lambda i: f"{i} — {cat_labels[i].replace(chr(10), ' ')}").to_string())

In [ ]:
X_hurdle = hurdle_encode(X_train)

hurdle_cols = ["MentHlth_any", "MentHlth_days", "PhysHlth_any", "PhysHlth_days"]
print("Neue Spalten nach hurdle_encode:")
print(X_hurdle[hurdle_cols].head(10).to_string(index=True))

print(f"\nOriginal-Spalten entfernt: {set(['MentHlth','PhysHlth']) - set(X_hurdle.columns)}")
print(f"Neue Spalten hinzugefügt:  {set(hurdle_cols) & set(X_hurdle.columns)}")
print(f"\nSpaltenzahl vorher: {X_train.shape[1]}  |  nachher: {X_hurdle.shape[1]}")

# Kurze Plausibilitätsprüfung: _any stimmt mit >0 überein
assert (X_hurdle["MentHlth_any"] == (X_hurdle["MentHlth_days"] > 0)).all(), \
    "MentHlth_any stimmt nicht mit MentHlth_days > 0 überein!"
assert (X_hurdle["PhysHlth_any"] == (X_hurdle["PhysHlth_days"] > 0)).all(), \
    "PhysHlth_any stimmt nicht mit PhysHlth_days > 0 überein!"
print("\nAssertion bestanden: _any-Flags konsistent mit _days-Werten.")